In [ ]:
import io
import sys
from pathlib import Path

import scipy
import numpy as np
import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy.sparse import hstack as sparse_hstack
from xgboost import XGBRegressor

sys.path.insert(0, str(Path.cwd().parent))

from model_wrapper import predict_pipeline

In [ ]:
STORAGE_ACCOUNT = "salaryprdata"
CONTAINER = "pipeline-data"

In [ ]:
def _blob_client(blob_name: str):
    credential = DefaultAzureCredential()
    url = f"https://{STORAGE_ACCOUNT}.blob.core.windows.net"
    service = BlobServiceClient(url, credential=credential)
    return service.get_blob_client(container=CONTAINER, blob=blob_name)

def blob_exists(blob_name: str) -> bool:
    return _blob_client(blob_name).exists()

def load_parquet_from_blob(blob_name: str) -> pd.DataFrame:
    buf = io.BytesIO(_blob_client(blob_name).download_blob().readall())
    return pd.read_parquet(buf)

In [ ]:

def evaluate(y_true, y_pred):
    """
    Calculate regression metrics.
    Returns dict with rmse, mae, r2.
    """
    return {
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }


def fit_pipeline(df, cat_vars, num_vars):
    """Fit all transformers and model on the given dataframe. Returns bundle."""

    title_tfidf = TfidfVectorizer(
        max_features=10_000, ngram_range=(1, 2), min_df=3, sublinear_tf=True
    )
    X_title = title_tfidf.fit_transform(df["title"].fillna(""))

    desc_tfidf = TfidfVectorizer(
        max_features=10_000, ngram_range=(1, 2), min_df=3, sublinear_tf=True
    )
    desc_svd = TruncatedSVD(n_components=100, random_state=42)
    X_desc = desc_svd.fit_transform(
        desc_tfidf.fit_transform(df["full_description"].fillna(""))
    )

    ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    X_structured = np.hstack(
        [
            ohe.fit_transform(df[cat_vars]),
            scaler.fit_transform(imputer.fit_transform(df[num_vars])),
        ]
    )

    X = sparse_hstack([X_title, scipy.sparse.csr_matrix(X_desc), scipy.sparse.csr_matrix(X_structured)])
    y = df["avg_salary"].values

    # model = Ridge(alpha=1.0)
    model = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=5, random_state=42, subsample=0.8,
        colsample_bytree=0.8, min_child_weight=10
        )
    
    model.fit(X, y)

    return (
        {
            "title_tfidf": title_tfidf,
            "desc_tfidf": desc_tfidf,
            "desc_svd": desc_svd,
            "ohe": ohe,
            "imputer": imputer,
            "scaler": scaler,
            "model": model,
            "cat_vars": cat_vars,
            "num_vars": num_vars,
        },
        X,
        y,
    )


def train_model():

    # --- Load and merge data ---
    # Try to load from blob storage first, fallback to local files
    try:
        df = load_parquet_from_blob("processed/feature_engineered_data.parquet")
        # limit df to only records created before March 27
        df = df[df["created"] < "2024-03-27"].reset_index(drop=True)
        descriptions = load_parquet_from_blob("raw/urls_with_descriptions.parquet")
        descriptions.rename(columns={"description": "full_description"}, inplace=True)
        print(f"Loaded {len(df)} records and {len(descriptions)} descriptions from blob storage")
    except Exception as e:
        print(f"ERROR: Failed to load data from blob storage: {e}")
        raise
    
    df = df.merge(
        descriptions[["redirect_url", "full_description"]],
        on="redirect_url",
        how="left",
    )
    df["full_description"] = df["full_description"].fillna("")
    df = df[df["full_description"].str.len() > 30]
    print(f"Loaded {len(df)} records.")

    cat_vars = [
        "contract_type",
        "contract_time",
        "category_label",
        "location_area_length",
        "location_state",
        "location_region_abridged",
        "location_city_abridged",
        "missing_long_lat",
    ]

    num_vars = ["longitude", "latitude"]

    # --- Time-based split ---
    df = df.sort_values("created").reset_index(drop=True)
    split_idx = int(len(df) * 0.8)
    train_df = df.iloc[:split_idx]
    val_df = df.iloc[split_idx:]
    print(f"Train: {len(train_df)} records, Val: {len(val_df)} records.")

    # --- Fit on train, evaluate on holdout ---
    print("Fitting on training set...")
    train_bundle, X_train, y_train = fit_pipeline(train_df, cat_vars, num_vars)

    y_train_pred = train_bundle["model"].predict(X_train)
    train_metrics = evaluate(y_train, y_train_pred)
    print(f"Train MAE: ${train_metrics['mae']:,.0f}")

    val_preds = predict_pipeline(train_bundle, val_df)
    val_metrics = evaluate(val_df["avg_salary"].values, val_preds)
    print(f"Val MAE: ${val_metrics['mae']:,.0f}")

    return train_metrics, val_metrics

In [ ]:
# Baseline parameters
train_model()

In [ ]:
# With XGBoost - depth of 6
train_model()

In [ ]:
# With XGBoost - depth of 5
train_model()

In [ ]:
# With XGBoost - depth of 6 with subsampling
train_model()

In [ ]:
# With XGBoost - depth of 6 with subsampling and min_child_weight = 5
train_model()

In [ ]:
# With XGBoost - depth of 6 with subsampling and min_child_weight = 10
train_model()

In [ ]:
df = load_parquet_from_blob("processed/feature_engineered_data.parquet")
df["avg_salary"].describe()

In [ ]:
df = df.sort_values("created").reset_index(drop=True)
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
val_df = df.iloc[split_idx:]

In [ ]:
train_df['avg_salary'].describe()

In [ ]:
val_df['avg_salary'].describe()

In [ ]:
train_df.shape, val_df.shape

In [ ]:
print(train_df["category_label"].value_counts(normalize=True).head(10))
print(val_df["category_label"].value_counts(normalize=True).head(10))

In [ ]:
# With XGBoost - depth of 6 with subsampling and min_child_weight = 5 and trigrams
train_model()